# Wheat Futures Data Builder

This notebook isolates the wheat-price ingestion task from the main project notebook.

Goals:
- probe what Yahoo Finance actually provides for continuous wheat futures history
- download `ZW=F` in chunks when requested
- save a canonical local CSV cache for the project notebook
- keep the final-project notebook on a simple CSV-only path by default


In [ ]:
from pathlib import Path
import os
import warnings

os.environ.setdefault('MPLCONFIGDIR', '/Users/jlaw/projects/stern/systematic-investing/.mplconfig')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

warnings.filterwarnings('ignore', category=FutureWarning)

PROJECT_ROOT = Path('/Users/jlaw/projects/stern/systematic-investing')
DATA_ROOT = PROJECT_ROOT / 'data' / 'ag_futures'
RAW_FUTURES_DIR = DATA_ROOT / 'raw' / 'futures'
PROCESSED_DIR = DATA_ROOT / 'processed'

RAW_FUTURES_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_START_DATE = '2000-02-18'
TARGET_END_DATE = pd.Timestamp.today().normalize().strftime('%Y-%m-%d')
CHUNK_YEARS = 3
REFRESH_WHEAT_FROM_YAHOO = False

RAW_CONTINUOUS_CACHE = RAW_FUTURES_DIR / 'zw_continuous_chunked_raw.csv'
PROCESSED_CONTINUOUS_CACHE = PROCESSED_DIR / 'wheat_continuous_back_adjusted.csv'

ROLL_MONTHS = [3, 5, 7, 9, 12]


In [ ]:
def normalize_yfinance_frame(frame):
    data = frame.copy()
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    keep = [col for col in ['Open', 'High', 'Low', 'Close', 'Volume'] if col in data.columns]
    return data[keep].dropna().copy()


def back_adjust_continuous_futures(frame):
    data = frame.copy()
    data['month'] = data.index.month
    data['is_roll_switch'] = data['month'].isin(ROLL_MONTHS) & (data['month'] != data['month'].shift(1))
    data['daily_change'] = data['Close'].diff()
    data['typical_change'] = (data['daily_change'].shift(1) + data['daily_change'].shift(-1)) / 2.0
    data['roll_gap'] = 0.0
    data.loc[data['is_roll_switch'], 'roll_gap'] = (
        data.loc[data['is_roll_switch'], 'daily_change'] - data.loc[data['is_roll_switch'], 'typical_change']
    )
    data['roll_gap'] = data['roll_gap'].fillna(0.0)
    data['cum_adjustment'] = data['roll_gap'].iloc[::-1].cumsum().iloc[::-1].shift(-1).fillna(0.0)
    data['Adj_Close_Futures'] = data['Close'] + data['cum_adjustment']
    data['futures_ret'] = data['Adj_Close_Futures'].pct_change()
    return data


def generate_chunk_windows(start_date, end_date, chunk_years=3):
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    windows = []
    cursor = start
    while cursor <= end:
        window_end = min(cursor + pd.DateOffset(years=chunk_years), end + pd.Timedelta(days=1))
        windows.append((cursor.normalize(), pd.Timestamp(window_end).normalize()))
        cursor = window_end
    return windows


def download_continuous_wheat_in_chunks(start_date, end_date, chunk_years=3):
    frames = []
    windows = generate_chunk_windows(start_date, end_date, chunk_years=chunk_years)
    print(f'Chunk windows: {len(windows)}')
    for idx, (window_start, window_end) in enumerate(windows, start=1):
        print(f'[{idx}/{len(windows)}] Downloading ZW=F from {window_start.date()} to {window_end.date()}')
        frame = yf.download(
            'ZW=F',
            start=window_start.strftime('%Y-%m-%d'),
            end=window_end.strftime('%Y-%m-%d'),
            auto_adjust=False,
            progress=False,
            actions=False,
        )
        frame = normalize_yfinance_frame(frame)
        if frame.empty:
            print('  -> empty chunk')
            continue
        print(f"  -> rows={len(frame)} first={frame.index.min().date()} last={frame.index.max().date()}")
        frames.append(frame)
    if not frames:
        raise RuntimeError('Yahoo returned no continuous wheat data across all requested windows.')
    combined = pd.concat(frames).sort_index()
    combined = combined[~combined.index.duplicated(keep='last')]
    return combined


In [ ]:
if RAW_CONTINUOUS_CACHE.exists() and not REFRESH_WHEAT_FROM_YAHOO:
    wheat_raw = pd.read_csv(RAW_CONTINUOUS_CACHE, parse_dates=['Date']).set_index('Date').sort_index()
    print(f'Loaded raw continuous wheat cache from {RAW_CONTINUOUS_CACHE}')
else:
    wheat_raw = download_continuous_wheat_in_chunks(TARGET_START_DATE, TARGET_END_DATE, chunk_years=CHUNK_YEARS)
    wheat_raw.to_csv(RAW_CONTINUOUS_CACHE, index_label='Date')
    print(f'Saved raw continuous wheat cache to {RAW_CONTINUOUS_CACHE}')

wheat = back_adjust_continuous_futures(wheat_raw)
wheat.to_csv(PROCESSED_CONTINUOUS_CACHE, index_label='Date')

print('Coverage summary')
print('rows', len(wheat))
print('first_date', wheat.index.min().date())
print('last_date', wheat.index.max().date())
print(f'Saved processed wheat cache to {PROCESSED_CONTINUOUS_CACHE}')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
wheat[['Close', 'Adj_Close_Futures']].plot(ax=axes[0], linewidth=1.5)
axes[0].set_title('Wheat continuous series: raw vs back-adjusted')
axes[0].grid(True, alpha=0.3)
roll_view = wheat.loc[wheat['is_roll_switch'], ['Close', 'Adj_Close_Futures', 'roll_gap']]
if not roll_view.empty:
    roll_view[['Close', 'Adj_Close_Futures']].plot(ax=axes[1], marker='o')
axes[1].set_title('Roll-switch checkpoints')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
